# 湖滨度假村预订取消预测 —— XGBoost 方案

## 商业问题
度假村提前几个月接受预订，约 **40%** 的预订最终没有兑现（取消）。营收团队想要一个标志：下赛季哪些预订会被取消？

## 任务与评分
- 目标：对 `test.csv` 中 **6,655** 条预订预测 `Canceled`（0=入住，1=取消）。
- 评分标准：**分类准确率**（accuracy）。
- 基线：完全不做任何分析，全部预测 0（没人取消），准确率约 **59.8%**。任何模型都必须超过它。

## 流程
1. 加载数据
2. 探索性分析（EDA）
3. 特征工程
4. 训练模型 + 按预订分组的交叉验证（GroupKFold，避免同一预订的多房间泄漏）
5. 阈值调优（准确率最优的决策阈值）
6. 预测测试集并生成 `submission.csv`


In [1]:
import sys
from pathlib import Path

# 让中文在 Windows 终端按 UTF-8 输出，避免 GBK 代码页乱码
try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score

# 数据与本 notebook 在同一目录
BASE = Path.cwd()
if not (BASE / "train.csv").exists():
    BASE = BASE / "9-21-2026"

train = pd.read_csv(BASE / "train.csv")   # 训练集：有 Canceled 标签
test = pd.read_csv(BASE / "test.csv")     # 测试集：无标签，需要预测
sample = pd.read_csv(BASE / "sample_submission.csv")

y = train["Canceled"].values

print("训练集形状：", train.shape)
print("测试集形状：", test.shape)
print("训练集取消率：%.4f" % y.mean())
print("全预测 0 的基线准确率：%.4f" % (1 - y.mean()))


训练集形状： (13544, 16)
测试集形状： (6655, 15)
训练集取消率：0.4022
全预测 0 的基线准确率：0.5978


In [2]:
# ---- EDA：目标分布与缺失值 ----
print("[缺失值]")
print(train.isnull().sum()[train.isnull().sum() > 0].to_string())
print()

# 分类特征与取消率的关系
for c in ["Channel", "DepositType", "LoyaltyTier", "GuestRegion",
          "ParkingRequested", "IsWeekendStay"]:
    g = train.groupby(c, dropna=False)["Canceled"].agg(["mean", "count"])
    g["mean"] = g["mean"].round(3)
    print(f"[{c}] 各取值取消率")
    print(g.to_string())
    print()


[缺失值]
DepositType        260
RoomRateUSD        446
SpecialRequests    283
LoyaltyTier        708

[Channel] 各取值取消率
              mean  count
Channel                  
Corporate    0.083   1572
Direct       0.296   3299
OTA          0.570   6261
TravelAgent  0.320   2412

[DepositType] 各取值取消率
                mean  count
DepositType                
NoDeposit      0.526   8227
NonRefundable  0.070   2024
Partial        0.290   3033
NaN            0.385    260

[LoyaltyTier] 各取值取消率
              mean  count
LoyaltyTier              
Gold         0.343    233
Silver       0.316   2017
Standard     0.421  10586
NaN          0.386    708

[GuestRegion] 各取值取消率
              mean  count
GuestRegion              
East         0.401   4102
North        0.401   2997
Overseas     0.414   1370
South        0.401   2760
West         0.400   2315

[ParkingRequested] 各取值取消率
                   mean  count
ParkingRequested              
False             0.400   9736
True              0.408   3808

[IsW

In [3]:
# ---- EDA：提前天数 / 历史取消次数 的强单调信号 ----
# LeadTimeDays 分箱：提前越久，取消率越高
train["lead_bin_tmp"] = pd.cut(
    train["LeadTimeDays"],
    bins=[-1, 7, 30, 60, 90, 120, 180, 1000],
    labels=["<=7", "8-30", "31-60", "61-90", "91-120", "121-180", ">180"],
)
print("[LeadTimeDays 分箱]")
print(train.groupby("lead_bin_tmp", observed=True)["Canceled"]
      .agg(["mean", "count"]).round(3).to_string())
print()

# PriorCancellations：历史取消越多，本次取消概率越高
print("[PriorCancellations]")
print(train.groupby("PriorCancellations")["Canceled"]
      .agg(["mean", "count"]).round(3).to_string())
print()

# BookingId 结构：B-<reservation>-<room>，同一预订的多个房间共用中间号
train["reservation"] = train["BookingId"].str.extract(r"B-(\d+)-")
res_size = train.groupby("reservation")["BookingId"].transform("count")
train["res_size_tmp"] = res_size.astype(int)
print("[预订内房间数 res_size]：3+ 间房的团订取消率明显更高")
print(train.groupby("res_size_tmp")["Canceled"]
      .agg(["mean", "count"]).round(3).to_string())


[LeadTimeDays 分箱]
               mean  count
lead_bin_tmp              
<=7           0.171    538
8-30          0.201   3393
31-60         0.326   3621
61-90         0.395   2437
91-120        0.668   1536
121-180       0.740   1352
>180          0.753    667

[PriorCancellations]
                     mean  count
PriorCancellations              
0                   0.374  12340
1                   0.460    298
2                   0.706    293
3                   0.795    312
4                   0.804    301

[预订内房间数 res_size]：3+ 间房的团订取消率明显更高
               mean  count
res_size_tmp              
1             0.311   4876
2             0.310   3408
3             0.557   2436
4             0.529   1664
5             0.550   1160


In [4]:
# ---- 特征工程 ----
def engineer(df):
    df = df.copy()

    # BookingId -> 预订号 / 房号 / 预订内房间数
    df["reservation"] = df["BookingId"].str.extract(r"B-(\d+)-")
    df["room_no"] = df["BookingId"].str.extract(r"-(\d+)$").astype(int)
    df["res_size"] = df.groupby("reservation")["BookingId"] \
        .transform("count").astype(int)

    # 数值派生特征
    df["total_rate"] = df["RoomRateUSD"] * df["Nights"]       # 总房费
    df["leadtime_roomrate"] = df["LeadTimeDays"] * df["RoomRateUSD"]

    # 布尔 -> 0/1
    df["IsWeekendStay"] = df["IsWeekendStay"].astype(int)
    df["ParkingRequested"] = df["ParkingRequested"].astype(int)

    # 分类特征缺失值 -> "missing"
    df["DepositType"] = df["DepositType"].fillna("missing")
    df["LoyaltyTier"] = df["LoyaltyTier"].fillna("missing")

    # 提前天数分箱（分段单调性很强）
    df["lead_bin"] = pd.cut(
        df["LeadTimeDays"],
        bins=[-1, 7, 30, 60, 90, 120, 180, 1000],
        labels=["<=7", "8-30", "31-60", "61-90", "91-120", "121-180", ">180"],
    ).astype(str)

    return df


train = engineer(train)
test = engineer(test)
groups = train["reservation"].values  # GroupKFold 分组依据

cat_features = ["Channel", "DepositType", "LoyaltyTier", "GuestRegion", "lead_bin"]
num_features = [
    "LeadTimeDays", "RoomRateUSD", "PartySize", "Nights", "ArrivalMonth",
    "SpecialRequests", "PriorStays", "PriorCancellations", "room_no",
    "res_size", "total_rate", "leadtime_roomrate",
    "IsWeekendStay", "ParkingRequested",
]
FEATURES = num_features + cat_features

# 分类特征统一为 category dtype，交给 XGBoost 原生处理
X_train = train[FEATURES].copy()
X_test = test[FEATURES].copy()
for c in cat_features:
    cats = pd.CategoricalDtype(
        categories=sorted(set(X_train[c]) | set(X_test[c]))
    )
    X_train[c] = X_train[c].astype(cats)
    X_test[c] = X_test[c].astype(cats)

print("特征数量：", len(FEATURES))
print("特征列表：", FEATURES)


特征数量： 19
特征列表： ['LeadTimeDays', 'RoomRateUSD', 'PartySize', 'Nights', 'ArrivalMonth', 'SpecialRequests', 'PriorStays', 'PriorCancellations', 'room_no', 'res_size', 'total_rate', 'leadtime_roomrate', 'IsWeekendStay', 'ParkingRequested', 'Channel', 'DepositType', 'LoyaltyTier', 'GuestRegion', 'lead_bin']


In [5]:
# ---- 模型训练 + GroupKFold 交叉验证 ----
# 用 GroupKFold 按“预订”分组划分：同一预订的房间不会同时出现在训练/验证集，
# 与真实测试集（全新预订）的评估方式一致，避免乐观偏差。

params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "max_depth": 5,
    "learning_rate": 0.04,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "min_child_weight": 10,
    "reg_lambda": 1.0,
    "n_estimators": 2500,
    "early_stopping_rounds": 100,
    "random_state": 0,
}

gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(y))
best_iters = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y, groups)):
    Xtr, Xva = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    ytr, yva = y[tr_idx], y[va_idx]

    model = xgb.XGBClassifier(**params, enable_categorical=True)
    model.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)

    p = model.predict_proba(Xva)[:, 1]
    oof[va_idx] = p
    best_iters.append(model.best_iteration)
    acc = accuracy_score(yva, (p >= 0.5).astype(int))
    print(f"fold {fold}: acc={acc:.4f}  best_iter={model.best_iteration}")

print()
print("OOF 准确率 @0.5 = %.4f" % accuracy_score(y, (oof >= 0.5).astype(int)))


fold 0: acc=0.8224  best_iter=166
fold 1: acc=0.8291  best_iter=284
fold 2: acc=0.8350  best_iter=211
fold 3: acc=0.8173  best_iter=194
fold 4: acc=0.8146  best_iter=204

OOF 准确率 @0.5 = 0.8237


In [6]:
# ---- 阈值调优：准确率最优的决策阈值 ----
# 类别约 40/60，默认阈值 0.5 未必最优，在 OOF 概率上搜索使准确率最大的阈值。
best_t, best_acc = 0.5, accuracy_score(y, (oof >= 0.5).astype(int))
for t in np.arange(0.40, 0.60, 0.01):
    a = accuracy_score(y, (oof >= t).astype(int))
    if a > best_acc:
        best_acc, best_t = a, t
print(f"最优阈值 = {best_t:.2f} -> OOF 准确率 = {best_acc:.4f}")


最优阈值 = 0.46 -> OOF 准确率 = 0.8260


In [7]:
# ---- 全量训练 + 预测 + 生成提交文件 ----
# 用各折 best_iteration 的中位数作为全量训练树数
full_params = {**params, "n_estimators": int(np.median(best_iters))}
full_params.pop("early_stopping_rounds", None)

full_model = xgb.XGBClassifier(**full_params, enable_categorical=True)
full_model.fit(X_train, y, verbose=False)

pred_prob = full_model.predict_proba(X_test)[:, 1]
pred = (pred_prob >= best_t).astype(int)

sub = sample.copy()
sub["Canceled"] = pred
sub.to_csv(BASE / "submission.csv", index=False)

print("已保存 submission.csv，共", len(sub), "行")
print("预测为取消的比例：%.4f" % pred.mean())
print("预测分布（0/1）：", np.bincount(pred))


已保存 submission.csv，共 6655 行
预测为取消的比例：0.3950
预测分布（0/1）： [4026 2629]


In [8]:
# ---- 特征重要性 ----
imp = pd.Series(full_model.feature_importances_, index=FEATURES) \
    .sort_values(ascending=False)
print(imp.round(4).to_string())


DepositType           0.2062
LeadTimeDays          0.1974
res_size              0.1116
PriorCancellations    0.1071
Channel               0.1007
lead_bin              0.0602
LoyaltyTier           0.0304
SpecialRequests       0.0284
Nights                0.0192
leadtime_roomrate     0.0187
PartySize             0.0157
ArrivalMonth          0.0156
RoomRateUSD           0.0147
total_rate            0.0136
room_no               0.0131
PriorStays            0.0122
IsWeekendStay         0.0120
GuestRegion           0.0118
ParkingRequested      0.0115


## 结果总结
- **OOF 准确率约 0.826**，显著超过全预测 0 的基线 0.598。
- 最强信号：`DepositType`（NonRefundable 几乎不取消）、`LeadTimeDays`（提前越久越易取消）、
  `res_size`（3+ 间房的团订更易取消）、`PriorCancellations`（历史取消越多本次越易取消）、`Channel`（OTA 最高、Corporate 最低）。
- 采用 GroupKFold（按预订分组）保证评估诚实，与线上测试集一致。
- 提交文件已生成：`submission.csv`（BookingId, Canceled）。
